# Markov Chain Monte Carlo (MCMC) Sampling

> Editor: Weipeng Xu
> 
> Last modified: 08/08/2025

For high-dimensional or complicated probability density functions, direct sampling to obatin a sequence of random sampled could be challenging due to **the intractability of normalization** and **the curse of dimensionality**. One commonly-used approach to tackle this challenge is the Markov chain Monte Carlo (MCMC) method, which constructs a Markov chain  with the stationary distribution being target distribution. Due to their effectiveness and flexibility, MCMC methods have been widely used in modern Bayesian inference.

## Markov Chain

A Markov chain is a stochastic process describing a sequence of possible events where the probability of possible events only depends on the state of the previous event.

Mathematically, a Markov chain is defined by a set of states $\{\boldsymbol{x}_1, \boldsymbol{x}_2, \ldots\}$, where each $\boldsymbol{x} \in \mathbb{R}^d$ is a $d$-dimensional vector. The Markov property (memorylessness) can be written as
$$
\tag{1}
\mathcal{P}(\boldsymbol{x}_{t+1} | \boldsymbol{x}_1, \boldsymbol{x}_2, \ldots, \boldsymbol{x}_t) = \mathcal{P}(\boldsymbol{x}_{t+1} | \boldsymbol{x}_t)
$$
where $\mathcal{P}(\boldsymbol{x}_{t+1} | \boldsymbol{x}_t)$ is the conditional probability of transitioning from $\boldsymbol{x}_t$ to $\boldsymbol{x}_{t+1}$. In MCMC, this conditional probability is often expressed via the transition kernel $K(\boldsymbol{x}' | \boldsymbol{x})$, which governs the evolution of the chain. For any measurable set $A \subseteq \mathbb{R}^d$,
$$
\tag{2}
\mathcal{P}(\boldsymbol{x}_{t+1} \in A \mid \boldsymbol{x}_t = \boldsymbol{x}) = \int_A K(\boldsymbol{x}' | \boldsymbol{x}) d\boldsymbol{x}'
$$
Here, $\boldsymbol{x}$ denotes the current state, and $\boldsymbol{x}'$ denotes a possible next state. The distribution of the next state $\boldsymbol{x}_{t+1}$ is then given by
$$
\tag{3}
p_{t+1}(\boldsymbol{x}') = \int K(\boldsymbol{x}' | \boldsymbol{x}) p_t(\boldsymbol{x}) \, d\boldsymbol{x}
$$
where $p_t(\boldsymbol{x})$ is the distribution at time $t$.

If the transition kernel $K(\boldsymbol{x}' | \boldsymbol{x})$ satisfies the **detailed balance condition** with respect to a distribution $\pi(\boldsymbol{x})$, i.e.,
$$
\tag{4}
\pi(\boldsymbol{x}) K(\boldsymbol{x}' | \boldsymbol{x}) = \pi(\boldsymbol{x}') K(\boldsymbol{x} | \boldsymbol{x}')
$$
then $\pi(\boldsymbol{x})$ is a stationary distribution of the Markov chain. In MCMC, we design the transition kernel so that the stationary distribution $\pi(\boldsymbol{x})$ is exactly the distribution we wish to sample from.

Different choices of the transition kernel $K(\boldsymbol{x}' | \boldsymbol{x})$ lead to different MCMC algorithms, which will be introduced in the next sections.

## Metropolis-Hastings (MH)

Metropolis-Hastings (MH) is one of the fundamental MCMC algorithms. To construct a kernel $K(\boldsymbol{x}'|\boldsymbol{x})$ that satisfies the detailed balance condition (4), MH introduces a proposal distribution $q(\boldsymbol{x}'|\boldsymbol{x})$ and an acceptance probability $\alpha(\boldsymbol{x}, \boldsymbol{x}')$. Specifically, the transition kernel is defined as
$$
\tag{5}
K(\boldsymbol{x}'|\boldsymbol{x}) = q(\boldsymbol{x}'|\boldsymbol{x}) \alpha(\boldsymbol{x}, \boldsymbol{x}') + \delta(\boldsymbol{x}' - \boldsymbol{x}) \left[1 - \int q(\boldsymbol{y}|\boldsymbol{x}) \alpha(\boldsymbol{x}, \boldsymbol{y}) d\boldsymbol{y}\right]
$$
where $\delta(\cdot)$ is the Dirac delta function. The acceptance probability is chosen as
$$
\tag{6}
\alpha(\boldsymbol{x}, \boldsymbol{x}') = \min\left[1, \frac{\pi(\boldsymbol{x}') q(\boldsymbol{x}|\boldsymbol{x}')}{\pi(\boldsymbol{x}) q(\boldsymbol{x}'|\boldsymbol{x})}\right]
$$

After proposing a candidate state $\boldsymbol{x}'$ from $q(\boldsymbol{x}'|\boldsymbol{x})$, a random number $u$ is drawn from the uniform distribution on $[0, 1]$, which is compared to the acceptance probability $\alpha(\boldsymbol{x}, \boldsymbol{x}')$:
- If $u < \alpha(\boldsymbol{x}, \boldsymbol{x}')$, the move to $\boldsymbol{x}'$ is accepted;
- If $u > \alpha(\boldsymbol{x}, \boldsymbol{x}')$, the chain remains at $\boldsymbol{x}$. 

The proportion of proposed moves that are accepted during sampling is defined as the **acceptance rate**, which reflects how efficiently the Markov chain explores the state space:

- A very low acceptance rate means most proposals are rejected and the chain mixes slowly; 
- A very high acceptance rate may indicate the chain is making only small moves and not exploring well. 

In practice, tuning the proposal distribution to achieve a moderate acceptance rate (often between 0.2 and 0.5) helps balance exploration and efficiency.

This stochastic acceptance step ensures that the detailed balance condition holds for every possible case:
$$
\tag{7}
\pi(\boldsymbol{x}) q(\boldsymbol{x}'|\boldsymbol{x}) \alpha(\boldsymbol{x}, \boldsymbol{x}') = \pi(\boldsymbol{x}') q(\boldsymbol{x}|\boldsymbol{x}') \alpha(\boldsymbol{x}', \boldsymbol{x})
$$
 and allows for both exploration and rejection, balancing local moves and global mixing.

We can see that the acceptance probability $\alpha(\boldsymbol{x}, \boldsymbol{x}')$ only involves the ratio of the target density $\pi$ at the proposed and current states, and the proposal probabilities. The normalization constant of $\pi$ cancels out in the ratio, so we do not need to know or compute it. This property makes MH especially powerful for Bayesian inference, where the target posterior is often only known up to a constant factor.

Furthermore, when the proposal distribution $q(\boldsymbol{x}'|\boldsymbol{x})$ is symmetric, i.e., $q(\boldsymbol{x}'|\boldsymbol{x}) = q(\boldsymbol{x}|\boldsymbol{x}')$, the acceptance probability simplifies to
$$
\tag{8}
\alpha(\boldsymbol{x}, \boldsymbol{x}') = \min\left[1, \frac{\pi(\boldsymbol{x}')}{\pi(\boldsymbol{x})}\right]
$$
A common and practical choice for symmetric proposal is the **Gaussian distribution**:
$$
\tag{9}
q(\boldsymbol{x}'|\boldsymbol{x}) = \mathcal{N}(\boldsymbol{x}'; \boldsymbol{x}, \Sigma)
$$
This leads to the so-called **Gaussian random walk Metropolis algorithm**, where each new candidate is proposed by adding Gaussian noise to the current state. The chain thus performs a random walk in the state space, and the efficiency of exploration depends on the choice of covariance $\Sigma$ and the structure of the target distribution.

The MH algorithm can be summarized as:

----------------
1. Initialize $\boldsymbol{x}^{(0)}$

2. For $t = 0, 1, \ldots, T-1$:
    - Propose $\boldsymbol{x}' \sim q(\boldsymbol{x}'|\boldsymbol{x}^{(t)})$
    - Compute acceptance probability:
      $
      \alpha(\boldsymbol{x}^{(t)}, \boldsymbol{x}') = \min\left[1, \frac{\pi(\boldsymbol{x}') q(\boldsymbol{x}^{(t)}|\boldsymbol{x}')}{\pi(\boldsymbol{x}^{(t)}) q(\boldsymbol{x}'|\boldsymbol{x}^{(t)})}\right]
      $
    - Draw $u \sim \mathrm{Uniform}(0, 1)$
    - If $u < \alpha(\boldsymbol{x}^{(t)}, \boldsymbol{x}')$:
        - Set $\boldsymbol{x}^{(t+1)} = \boldsymbol{x}'$

      Else:
        - Set $\boldsymbol{x}^{(t+1)} = \boldsymbol{x}^{(t)}$
3. End For
----------------

## Hamiltonian Monte Carlo (HMC)

Hamiltonian Monte Carlo (HMC) is motivated by concepts from Hamiltonian dynamics in physics. Instead of relying on random walk proposals like MH. which usually suffers from slow mixing and high autocorrelation issues, HMC considers the sampling process as **simulating the trajectory of a particle moving in a phase space**.

To achieve this, HMC introduces an auxiliary momentum variable $\boldsymbol{p}$ for each position variable $\boldsymbol{x}$ (i.e., sampling variables). The state in the expanded space (phase space) is described by $(\boldsymbol{x}, \boldsymbol{p})$, and the joint probability distribution on the phase space is called the **canonical distribution**:
$$
\tag{10}
\pi(\boldsymbol{x}, \boldsymbol{p}) = \pi(\boldsymbol{p}|\boldsymbol{x})\pi(\boldsymbol{x})
$$
Due to the nature of the Hamiltonian system, the canconical density can be further written in terms of an invariant **Hamiltonian function** $H(\boldsymbol{x},\boldsymbol{p})$:
$$
\tag{11}
\pi(\boldsymbol{x}, \boldsymbol{p}) = \exp({-H(\boldsymbol{x}, \boldsymbol{p})})
$$
The $H(\boldsymbol{x}, \boldsymbol{p})$ (also called the **energy**) captures the geometry of the **typical set** of the phase space distribution, which can be decomposed as:
$$
\tag{12}
\begin{aligned}
H(\boldsymbol{x}, \boldsymbol{p}) &= -\log\pi(\boldsymbol{x}, \boldsymbol{p})\\
&=-\log\pi(\boldsymbol{p}|\boldsymbol{x})-\log\pi(\boldsymbol{x})\\
&=K(\boldsymbol{x}, \boldsymbol{p}) + U(\boldsymbol{x})
\end{aligned}
$$
The potential energy 
$$
U(\boldsymbol{x})=-\log\pi(\boldsymbol{x})
$$
corresponds to the density of the target distribution, while the kinetic energy $K(\boldsymbol{x}, \boldsymbol{p})$ is unconstrained and must be specified. A common choice for the conditional distribution over the momentum is a Gaussian distribution centered at $0$:
$$
\tag{13}
\pi(\boldsymbol{p}|\boldsymbol{x}) = \mathcal{N}(\boldsymbol{0}, \boldsymbol{M})
$$
which leads to a Euclidean-Gaussian kinetic energy defined as:
$$
\tag{14}
K(\boldsymbol{x},\boldsymbol{p}) = \frac{1}{2}\boldsymbol{p}^{\text{T}}\boldsymbol{M}^{-1}\boldsymbol{p}
$$ 
where the Euclidean metric is also known as the **mass matrix**. Theoretically, $M$ should match the covariance structure of the target distribution. This ensures that the Hamiltonian dynamics can efficiently explore all directions in the state space, with step sizes adapted to the actual scale and correlation of each dimension. In practice, $M$ is often chosen as:
- The identity matrix $M = I$, suitable for problems where all dimensions have similar scale and are weakly correlated.
- A diagonal matrix $M = \mathrm{diag}(\sigma_1^2, \ldots, \sigma_d^2)$, where each $\sigma_i^2$ is an estimate of the variance of the $i$-th dimension, improving efficiency for problems with different scales.
- The full covariance matrix $M = \Sigma$, estimated from warm-up samples or prior knowledge, which is most effective for strongly correlated or anisotropic target distributions.

Since the Hamilton function $H(\boldsymbol{x}, \boldsymbol{p})$ captures the geometry of the typical set, we can utilize it to generate a vector field to move thorugh the constant-energy hypersurface in the phase space with the following Hamilton's equations:
$$
\begin{aligned}
\tag{15}
&\frac{d\boldsymbol{x}}{dt} = \frac{\partial H}{\partial \boldsymbol{p}}=\frac{\partial K}{\partial \boldsymbol{p}}\\
&\frac{d\boldsymbol{p}}{dt} = -\frac{\partial H}{\partial \boldsymbol{x}}=-\frac{\partial K}{\partial \boldsymbol{x}}-\frac{\partial U}{\partial \boldsymbol{x}}
\end{aligned}
$$
which are often numerically solved using the **symplectic integrators**. A common choice is the **leapfrog integrator**: Given a time step size $\epsilon$ and current position $\boldsymbol{x}$, the trajectory can be computed by:
1. Half-step update of momentum:
$$
\tag{16}
\boldsymbol{p} \leftarrow \boldsymbol{p} - \frac{\epsilon}{2} \nabla_{\boldsymbol{x}} U(\boldsymbol{x})
$$
2. Full-step update of position:
$$
\tag{17}
\boldsymbol{x} \leftarrow \boldsymbol{x} + \epsilon \nabla_{\boldsymbol{p}} K(\boldsymbol{x},\boldsymbol{p})
$$
3. Half-step update of momentum:
$$
\tag{18}
\boldsymbol{p} \leftarrow \boldsymbol{p} - \frac{\epsilon}{2} \nabla_{\boldsymbol{x}} U(\boldsymbol{x})
$$
By repeating these steps $L$ times, the leapfrog integrator generates a proposal $(\boldsymbol{x}^*, \boldsymbol{p}^*)$ that approximately follows the true Hamiltonian trajectory, preserving volume and energy up to numerical error. Then a similar Metropolis acceptance step is adopted to ensure the samples follow the correct target distribution, and the acceptance probability is computed as:
$$
\tag{19}
\alpha = \min\left[1, \frac{\exp(-H(\boldsymbol{x}^*, \boldsymbol{p}^*))}{\exp(-H(\boldsymbol{x}, \boldsymbol{p}))}\right]
$$
Typically, the acceptance rate is targeted between 60% and 90%. After obatining the sample $(\boldsymbol{x}, \boldsymbol{p})$ in the phase space, we marginalize over the momentum variable to recover samples from the target distribution $\pi(\boldsymbol{x})$:
$$
\tag{20}
\pi(\boldsymbol{x}) = \int \pi(\boldsymbol{x}, \boldsymbol{p}) \, d\boldsymbol{p}
$$
For the commonly used standard Gaussian $\pi(\boldsymbol{p}|\boldsymbol{x})=\mathcal{N}(\boldsymbol{0}, \boldsymbol{M})$, this marginalization is straightforward
$$
\tag{21}
\pi(\boldsymbol{x}) = \pi(\boldsymbol{x}) \int \mathcal{N}(\boldsymbol{p}| \boldsymbol{0}, \boldsymbol{M}) \, d\boldsymbol{p} = \pi(\boldsymbol{x})
$$
which means that by marginalizing out the momentum variable, the resulting $\boldsymbol{x}$ samples indeed follow the target distribution. Therefore, in each iteration, we only need to retain the position variable $\boldsymbol{x}$, and the momentum $\boldsymbol{p}$ is discarded.

Due to the time-reversibility and volume-preserving nature of Hamiltonian dynamics, together with the Metropolis acceptance step that enforces balance for the joint distribution, it can be rigorously proven that sampling in phase space and marginalizing over the momentum variable yields a Markov chain in parameter space that satisfies the detailed balance condition for the target distribution $\pi(\boldsymbol{x})$.

The HMC algorithm can be summarized as:

----------------
1. Initialize $\boldsymbol{x}^{(0)}$

2. For $t = 0, 1, \ldots, T-1$:
    - Sample momentum $\boldsymbol{p} \sim \pi(\boldsymbol{p}|\boldsymbol{x})$
    - Set $(\boldsymbol{x}, \boldsymbol{p}) = (\boldsymbol{x}^{(t)}, \boldsymbol{p})$
    - Simulate Hamiltonian dynamics for $L$ steps with step size $\epsilon$ using the symplectic integrator to obtain proposal $(\boldsymbol{x}^*, \boldsymbol{p}^*)$
    - Compute acceptance probability:
      $
      \alpha = \min\left[1, \frac{\exp(-H(\boldsymbol{x}^*, \boldsymbol{p}^*))}{\exp(-H(\boldsymbol{x}, \boldsymbol{p}))}\right]
      $
    - Draw $u \sim \mathrm{Uniform}(0, 1)$
    - If $u < \alpha$:
        - Set $\boldsymbol{x}^{(t+1)} = \boldsymbol{x}^*$

      Else:
        - Set $\boldsymbol{x}^{(t+1)} = \boldsymbol{x}^{(t)}$
    - Marginalize over the momentum to obatin target samples
3. End For
----------------

## No-U-Turn Sampler (NUTS)

HMC is a powerful sampling method, but suffered from manually tuning both the step size ($\epsilon$) and the trajectory length ($L$ or total integration time $L\epsilon$). If $\epsilon$ is too large, the numerical integrator deviates from the true Hamiltonian trajectory, resulting in low acceptance rates and biased samples; if it is too small, the chain mixes slowly and computation is wasted on highly correlated samples. Similarly, setting the trajectory length is problematic: Too short a trajectory leads to highly correlated samples; too long increases computational cost without improving mixing. As a result, extensive trial-and-error and domain expertise are often required to tune HMC for different target distributions.

The No-U-Turn Sampler (NUTS) fundamentally improves HMC by introducing three mechanisms: automatic trajectory length control via the U-turn criterion, adaptive step size selection using dual averaging, and adaptive mass matrix ($\boldsymbol{M}$) estimation.

**1. U-turn Criterion and Trajectory Expansion (Detailed)**
NUTS eliminates manual trajectory length tuning by recursively constructing a binary tree of states along the Hamiltonian trajectory. Starting from $(\boldsymbol{x}_0, \boldsymbol{p}_0)$, NUTS alternately expands the trajectory in both forward and backward directions using the leapfrog integrator. At each recursion depth $j$, the trajectory doubles in length, and the sampler maintains two endpoints: $(\boldsymbol{x}_-, \boldsymbol{p}_-)$ for the backward end and $(\boldsymbol{x}_+, \boldsymbol{p}_+)$ for the forward end. The full trajectory is represented as a tree, where each node corresponds to a leapfrog step.

At each expansion, NUTS checks the U-turn criterion:
$$
(\boldsymbol{x}_+ - \boldsymbol{x}_-)^T \boldsymbol{p}_+ < 0 \quad \text{or} \quad (\boldsymbol{x}_+ - \boldsymbol{x}_-)^T \boldsymbol{p}_- < 0
$$
If either condition is satisfied, the trajectory is considered to have doubled back on itself, and further expansion is halted. This prevents the sampler from wasting computation on redundant integration and ensures efficient exploration.

During tree construction, NUTS collects all valid states $(\boldsymbol{x}_i, \boldsymbol{p}_i)$ that satisfy the slice condition:
$$
\exp(-H(\boldsymbol{x}_i, \boldsymbol{p}_i)) > \nu
$$
where $\nu \sim \mathrm{Uniform}(0, \exp(-H(\boldsymbol{x}_0, \boldsymbol{p}_0)))$ is a threshold drawn at the start of each iteration. The binary tree structure allows NUTS to efficiently traverse and sample from all candidate states generated before the U-turn. The next sample $\boldsymbol{x}_{\text{next}}$ is chosen uniformly (or multinomially weighted) from these candidates, ensuring unbiased sampling and detailed balance.

**2. Dual Averaging for Adaptive Step Size (In-Depth)**
During the warm-up phase, NUTS uses dual averaging to adapt the leapfrog step size $\epsilon$ for optimal efficiency. The goal is to achieve a target acceptance probability $\delta$ (typically $0.65$ or $0.8$). The algorithm maintains a running average of the acceptance probability $\hat{\alpha}_t$ and updates the log step size as follows:
$$
\log \epsilon_t = \mu - \frac{\sqrt{t}}{\gamma} (\hat{\alpha}_t - \delta)
$$
where $\mu$ is the log of the initial guess for step size, $\gamma$ is a learning rate parameter, and $t$ is the current iteration. The running average is updated recursively:
$$
\hat{\alpha}_t = (1 - \eta_t) \hat{\alpha}_{t-1} + \eta_t \alpha_t
$$
where $\alpha_t$ is the acceptance probability at iteration $t$, and $\eta_t$ is a diminishing adaptation rate, often set as $\eta_t = t^{-\kappa}$ for $\kappa \in (0.5, 1]$. The dual averaging scheme also maintains a running mean of the log step size for numerical stability:
$$
\bar{\log \epsilon}_t = (1 - \eta_t) \bar{\log \epsilon}_{t-1} + \eta_t \log \epsilon_t
$$
After the warm-up phase, the final step size is set to $\exp(\bar{\log \epsilon}_T)$, where $T$ is the total number of warm-up iterations. This ensures the step size is robust to early fluctuations and adapts smoothly to the target acceptance rate.

**3. Adaptive Mass Matrix ($\boldsymbol{M}$) Estimation (In-Depth)**
The mass matrix $\boldsymbol{M}$ determines the geometry of the momentum distribution:
$$
\pi(\boldsymbol{p}|\boldsymbol{x}) = \mathcal{N}(\boldsymbol{0}, \boldsymbol{M})
$$
During warm-up, NUTS estimates $\boldsymbol{M}$ by accumulating the empirical covariance of the position samples:
$$
\boldsymbol{M}_t = \frac{1}{t} \sum_{i=1}^t (\boldsymbol{x}_i - \bar{\boldsymbol{x}})(\boldsymbol{x}_i - \bar{\boldsymbol{x}})^T
$$
where $\bar{\boldsymbol{x}} = \frac{1}{t} \sum_{i=1}^t \boldsymbol{x}_i$ is the running mean. For numerical stability, a small regularization term $\lambda I$ is often added:
$$
\boldsymbol{M}_t^{\text{reg}} = \boldsymbol{M}_t + \lambda I
$$
where $I$ is the identity matrix and $\lambda$ is a small constant (e.g., $10^{-3}$). Depending on the problem, $\boldsymbol{M}$ can be estimated as a diagonal matrix (only variances) or a full covariance matrix (including correlations). After warm-up, $\boldsymbol{M}$ is fixed and used for all subsequent sampling steps, allowing the leapfrog integrator to efficiently explore directions with different scales and correlations.

**Final Sampling**
After trajectory expansion, step size and mass matrix adaptation, only the position variable is retained. The marginal distribution is
$$
\pi(\boldsymbol{x}) = \int \exp(-H(\boldsymbol{x}, \boldsymbol{p})) d\boldsymbol{p}
$$
NUTS thus achieves robust, efficient sampling by adaptively controlling trajectory length, integration accuracy, and momentum geometry, without manual tuning.

## References

1. Betancourt, Michael. "A conceptual introduction to Hamiltonian Monte Carlo." arXiv preprint arXiv:1701.02434 (2017).
2. https://www.georgeho.org/bayesian-inference-reading/
3. https://colindcarroll.com/blog/hmc_from_scratch.html
4. https://en.wikipedia.org/wiki